# Transformers playground

Run the two setup cells, then jump to any part.

1. **The GPU**
2. **Hugging Face**
3. **Tokens**
4. **ModernBERT**, a word in context
5. **One vector per passage**, and search
6. **CLIP**, inside and out
7. **API keys and LLM annotation**

Free Colab runs all of it.

> **File → Save a copy in Drive** first. Colab wipes its disk when the runtime ends.

In [ ]:
%pip install -q -U transformers sentence-transformers google-generativeai

In [ ]:
import os, re, time
import requests
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

from transformers import logging as hf_logging
hf_logging.set_verbosity_error()                 # quieten the load reports

def fetch(url, tries=7):
    """Download a URL, waiting and retrying. Big public archives throttle."""
    for n in range(tries):
        try:
            r = requests.get(url, timeout=120,
                             headers={"User-Agent": "culture-as-data course notebook"})
            if r.status_code == 200:
                return r
        except requests.exceptions.RequestException:
            pass
        if n == tries - 1:
            raise RuntimeError(f"gave up on {url}")
        time.sleep(3 * 2 ** n)

print("torch", torch.__version__)

## 1 · The GPU

A GPU does thousands of multiplications at once, which is all a transformer does.

Colab: **Runtime → Change runtime type → T4 GPU**.

In [ ]:
if torch.cuda.is_available():
    DEVICE = "cuda"
    print("GPU:", torch.cuda.get_device_name(0),
          round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
elif torch.backends.mps.is_available():          # Apple silicon, running locally
    DEVICE = "mps"
    print("Apple GPU")
else:
    DEVICE = "cpu"
    print("No GPU. Everything still works, slower.")
print("device:", DEVICE)

In [ ]:
# One big matrix multiply, timed on each device you have.
def time_matmul(device, n=2000, reps=5):
    a = torch.randn(n, n, device=device)
    b = torch.randn(n, n, device=device)
    if device == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(reps):
        a @ b
    if device == "cuda":
        torch.cuda.synchronize()
    return (time.time() - t0) / reps

cpu = time_matmul("cpu")
print(f"cpu   {cpu*1000:7.0f} ms")
if DEVICE != "cpu":
    gpu = time_matmul(DEVICE)
    print(f"{DEVICE:5s} {gpu*1000:7.0f} ms   ({cpu/gpu:.0f}x faster)")

## 2 · Hugging Face

Models live at [huggingface.co](https://huggingface.co/models), named `account/model`. Read the
model card. `pipeline` downloads the model, the tokenizer and the code around both.

In [ ]:
from transformers import pipeline

sentiment = pipeline("sentiment-analysis", device=DEVICE)
for line in ["The sequel was two hours I will never get back.",
             "I have not stopped thinking about the last twenty minutes.",
             "It was, I suppose, a film."]:
    out = sentiment(line)[0]
    print(f"{out['label']:8s} {out['score']:.2f}  {line}")

print("\nyou just used:", sentiment.model.config._name_or_path)

You did not pick that model, and it was trained on product reviews. Do you agree with the
third line?

In [ ]:
# Where the downloads go. Delete this folder to reclaim the disk.
from pathlib import Path
cache = Path(os.environ.get("HF_HOME", Path.home() / ".cache/huggingface"))
print(cache)
if cache.exists():
    size = sum(f.stat().st_size for f in cache.rglob("*") if f.is_file())
    print(f"{size/1e9:.2f} GB cached")

## 3 · Tokens

A model sees **tokens**, not letters: common words whole, rare ones in pieces.

In [ ]:
from transformers import AutoTokenizer

MODEL = "answerdotai/ModernBERT-base"
tok = AutoTokenizer.from_pretrained(MODEL)

for line in ["I ate the whole pizza.",
             "antidisestablishmentarianism",
             "Nosferatu, Whitby, Bistritz",
             "https://example.com/page?id=42"]:
    pieces = [p.replace("\u0120", " ").strip() for p in tok.tokenize(line)]
    print(f"{len(pieces):3d} | " + " / ".join(pieces))

In [ ]:
# Tokens are numbers. This is the actual input to every model below.
enc = tok("A museum keeps its collection.")
print(enc["input_ids"])
print(tok.convert_ids_to_tokens(enc["input_ids"]))

## 4 · ModernBERT

**ModernBERT** (2024), a rebuilt BERT: 22 layers, 8,192 tokens of context. An *encoder*, so it
reads rather than writes, returning one vector per token.

It was trained by filling in blanks.

In [ ]:
fill = pipeline("fill-mask", model=MODEL, device=DEVICE)

for s in ["I put the milk in the [MASK].",
          "The best thing about Chicago is the [MASK].",
          "Mary Shelley wrote [MASK] in 1818.",
          "My code failed because I forgot a [MASK]."]:
    print(f"{s:46s} {[g['token_str'].strip() for g in fill(s, top_k=4)]}")

The Shelley line is grammatical, not true.

### The same word, four sentences

In [ ]:
from transformers import AutoModel

model = AutoModel.from_pretrained(MODEL).to(DEVICE).eval()

def token_span(sentence, word):
    """Where `word` sits in the token list."""
    enc = tok(sentence, return_tensors="pt")
    ids = enc["input_ids"][0].tolist()
    want = tok(" " + word, add_special_tokens=False)["input_ids"]
    at = [i for i in range(len(ids) - len(want) + 1) if ids[i:i+len(want)] == want]
    if not at:
        raise ValueError(f"'{word}' not found as a whole token in: {sentence}")
    return enc, at[0], len(want)

def word_vector(sentence, word):
    enc, i, n = token_span(sentence, word)
    with torch.no_grad():
        states = model(**enc.to(DEVICE)).last_hidden_state[0]
    v = states[i:i+n].mean(0)
    return (v / v.norm()).cpu().numpy()

sentences = ["A bat flew out of the cave at dusk.",
             "The bat hung upside down all winter.",
             "She swung the bat and missed.",
             "He gripped the bat with both hands."]
V = np.stack([word_vector(s, "bat") for s in sentences])

print("     " + "".join(f"   s{i+1}" for i in range(4)))
for i, row in enumerate(V @ V.T):
    print(f"  s{i+1} " + "".join(f" {x:+.2f}" for x in row) + "   " + sentences[i])

Animals together, baseball bats together, the pairs apart. GloVe gives *bat* one vector. This
gives it one per sentence.

Every number is high, because raw encoder vectors all point roughly the same way. Read the
blocks, not the values. Part 5 fixes that.

## 5 · One vector per passage

To compare passages you need one vector each. Three ways:

- **CLS**, the first token.
- **Mean**, the average of the token vectors.
- **A trained sentence model**, fine-tuned so its vectors are comparable.

Test: 240 passages from *Dracula* and *Frankenstein*, downloaded from Project Gutenberg.
Nearest neighbour, same book?

In [ ]:
# Straight from Project Gutenberg. Public domain, no key, cached in ./downloads.
os.makedirs("downloads", exist_ok=True)

def gutenberg(book_id):
    path = f"downloads/{book_id}.txt"
    if not os.path.exists(path):
        url = f"https://www.gutenberg.org/cache/epub/{book_id}/pg{book_id}.txt"
        open(path, "w", encoding="utf-8").write(fetch(url).text)
    return open(path, encoding="utf-8", errors="ignore").read()

def passages(book_id, book, target=120):
    raw = gutenberg(book_id)
    a, b = raw.find("*** START OF"), raw.find("*** END OF")
    body = raw[raw.find("\n", a) + 1:b]
    paras = [" ".join(p.split()) for p in re.split(r"\n\s*\n", body)]
    paras = [p for p in paras if len(p.split()) >= 60]
    step = max(1, len(paras) // target)
    return [(book, p[:900]) for p in paras[::step]][:target]

rows = passages(345, "Dracula") + passages(84, "Frankenstein")
books = np.array([b for b, _ in rows])
texts = [t for _, t in rows]
print(len(rows), "passages")
print(texts[3][:120], "...")

In [ ]:
def pool(texts, how, batch=16):
    out = []
    for i in range(0, len(texts), batch):
        enc = tok(texts[i:i+batch], return_tensors="pt", padding=True,
                  truncation=True, max_length=256).to(DEVICE)
        with torch.no_grad():
            h = model(**enc).last_hidden_state
        m = enc["attention_mask"].unsqueeze(-1)
        v = h[:, 0] if how == "cls" else (h * m).sum(1) / m.sum(1)
        out.append(v.float().cpu().numpy())
    V = np.vstack(out)
    return V / np.linalg.norm(V, axis=1, keepdims=True)

from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("all-MiniLM-L6-v2", device=DEVICE)

ways = {
    "CLS, untrained":  pool(texts, "cls"),
    "mean, untrained": pool(texts, "mean"),
    "MiniLM, trained": embedder.encode(texts, normalize_embeddings=True, batch_size=32),
}
for name, V in ways.items():
    S = V @ V.T
    np.fill_diagonal(S, -9)
    right = (books[S.argmax(1)] == books).mean()
    off = S[np.triu_indices(len(V), 1)]
    print(f"{name:17s} neighbour from same book {right:.0%}   "
          f"cosines {off.min():+.2f} to {off.max():+.2f}")

Read the range, not the accuracy. Untrained, unrelated paragraphs score 0.96 and so do
near-identical ones. Trained, the number means something.

In [ ]:
S = ways["MiniLM, trained"]

def search(question, k=3):
    scores = S @ embedder.encode([question], normalize_embeddings=True)[0]
    for i in np.argsort(-scores)[:k]:
        print(f"  {scores[i]:.2f} {books[i]:13s} {' '.join(texts[i].split()[:16])}...")

for q in ["someone climbs down a wall",
          "a creature asks to be less alone",
          "the weather turns and the ship is in danger",
          "reading a letter from home"]:
    print("\n" + q)
    search(q)

None of those words appear in the passages.

In [ ]:
search("write your own question here", k=3)

In [ ]:
from sklearn.decomposition import PCA

pts = PCA(n_components=2).fit_transform(S)
plt.figure(figsize=(7, 5))
for book, colour in [("Dracula", "#A34526"), ("Frankenstein", "#2E6E8E")]:
    m = books == book
    plt.scatter(pts[m, 0], pts[m, 1], s=16, color=colour, alpha=0.75, label=book)
plt.xticks([]); plt.yticks([])
plt.legend(frameon=False)
plt.title("240 passages, arranged by meaning", loc="left")
plt.tight_layout(); plt.show()

Nobody told it which book. The overlap in the middle is correct.

## 6 · CLIP

400 million picture–caption pairs: pull each picture towards its own caption, push it from
everyone else's. Pictures and sentences end up in one space.

The 18 paintings below come straight from the Met's API, CC0 and no key needed. They have no
tags, only titles, which the model never sees. All 18 are portraits, so ask portrait questions.

In [ ]:
# Straight from the Met Collection API. No key. All CC0, cached in ./downloads.
MET = "https://collectionapi.metmuseum.org/public/collection/v1/objects/"
OBJECTS = [436532, 437397, 436986, 435896, 436896, 436544, 436295, 436623, 435802, 435581, 435944, 437390, 436658, 437530, 437874, 436840, 437055, 437510]

os.makedirs("downloads", exist_ok=True)
paintings, titles, made = [], [], []
for oid in OBJECTS:
    meta = fetch(MET + str(oid)).json()
    path = f"downloads/{oid}.jpg"
    if not os.path.exists(path):
        open(path, "wb").write(fetch(meta["primaryImageSmall"]).content)
    paintings.append(Image.open(path).convert("RGB"))
    titles.append(meta["title"])
    made.append(meta["objectDate"])          # the Met's own date, used in Part 7

clip = SentenceTransformer("clip-ViT-B-32", device=DEVICE)
pictures = clip.encode(paintings, normalize_embeddings=True)
print(len(paintings), "paintings,", pictures.shape[1], "numbers each")

In [ ]:
def look_for(phrase, k=3):
    scores = pictures @ clip.encode([phrase], normalize_embeddings=True)[0]
    best = np.argsort(-scores)[:k]
    fig, axes = plt.subplots(1, k, figsize=(3.4 * k, 3.4))
    for ax, i in zip(axes, best):
        ax.imshow(paintings[i])
        ax.set_title(f"{scores[i]:.2f}  {titles[i][:28]}", loc="left", fontsize=9)
        ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(f'"{phrase}"', x=0.02, ha="left", fontsize=12)
    plt.tight_layout(); plt.show()

look_for("a man in a straw hat")
look_for("a woman in a white headdress")
look_for("an artist at work")

Right answer every time, at 0.25 to 0.32. CLIP's scores sit in a narrow band; only the ranking
matters.

### Labels with no training

Labels as sentences, nearest one wins. No examples, no fitting. Edit the list and rerun.

In [ ]:
LABELS = ["a painting of a man", "a painting of a woman", "a landscape",
          "a religious painting", "a still life"]

scores = pictures @ clip.encode(LABELS, normalize_embeddings=True).T
for i, t in enumerate(titles):
    j = int(np.argmax(scores[i]))
    print(f"{LABELS[j]:26s} {scores[i, j]:.2f}   {t[:44]}")

Bronzino's *Portrait of a Young Man* comes out as woman, at 0.32, the same score as the ones it
gets right. The score is confidence, not correctness.

### What the picture looks like to it

A **Vision Transformer** cuts the image into a 7×7 grid of 32-pixel patches and treats each as a
token. Patches are to an image what tokens are to a sentence.

In [ ]:
from transformers import CLIPModel, CLIPProcessor

CLIP_ID = "openai/clip-vit-base-patch32"
vit = CLIPModel.from_pretrained(CLIP_ID, attn_implementation="eager").to(DEVICE).eval()
proc = CLIPProcessor.from_pretrained(CLIP_ID)

PICK = 0                                          # change this
img = paintings[PICK]
px = proc(images=img, return_tensors="pt")["pixel_values"]
square = np.array(img.resize((224, 224)))

fig, (a, b) = plt.subplots(1, 2, figsize=(9, 4.6))
a.imshow(square); a.set_title("what you see", loc="left", fontsize=10)
b.imshow(square)
for g in range(0, 225, 32):
    b.axvline(g, color="w", lw=1.2); b.axhline(g, color="w", lw=1.2)
b.set_title("what it sees: 49 patches", loc="left", fontsize=10)
for ax in (a, b):
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()
print(px.shape, "->", 224 // 32, "x", 224 // 32, "patches + 1 CLS token")

In [ ]:
# Every patch on its own. This is the whole input, in order.
patches = square.reshape(7, 32, 7, 32, 3).transpose(0, 2, 1, 3, 4)
fig, axes = plt.subplots(7, 7, figsize=(5.6, 5.6))
for i, ax in enumerate(axes.ravel()):
    ax.imshow(patches[i // 7, i % 7]); ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("49 tokens", x=0.02, ha="left", fontsize=12)
plt.tight_layout(); plt.show()

### Where it looks

The CLS token becomes the image's vector. Its attention row shows what fed into it.

In [ ]:
with torch.no_grad():
    att = vit.vision_model(px.to(DEVICE), output_attentions=True).attentions

def cls_map(layer):
    a = att[layer][0].mean(0)[0, 1:]              # average the heads, CLS row, drop CLS itself
    return a.reshape(7, 7).float().cpu().numpy()

# Dim each patch by how little the CLS token looked at it.
fig, axes = plt.subplots(1, 4, figsize=(12, 3.4))
axes[0].imshow(square / 255); axes[0].set_title("image", loc="left", fontsize=10)
for ax, layer in zip(axes[1:], (0, 5, 11)):
    m = cls_map(layer)
    lit = np.kron(m / m.max(), np.ones((32, 32)))[..., None]
    ax.imshow(square / 255 * (0.15 + 0.85 * lit))
    ax.set_title(f"layer {layer}", loc="left", fontsize=10)
for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("where the CLS token looked", x=0.02, ha="left", fontsize=12)
plt.tight_layout(); plt.show()

for layer in (0, 5, 11):
    m = cls_map(layer)
    print(f"layer {layer:2d}  busiest patch gets {m.max()/m.mean():.1f}x the average")

Flat at layer 0. By layer 11 a few patches carry several times the average, mostly the face and
the hat.

Attention shows where information came from, not why: a hypothesis, not evidence. See Jain and
Wallace, *Attention is not Explanation* (2019).

## 7 · API keys and LLM annotation

The models above run on your machine. A chat model like Gemini runs on Google's, and you
reach it with an **API key**: a password tied to your account that meters what you use.

Three rules. Never paste the key into a cell. Never commit it to GitHub. Store it in Colab's
**Secrets** panel (the key icon on the left) under the name `GEMINI_API_KEY`, with notebook
access switched on. A free key comes from [aistudio.google.com](https://aistudio.google.com/app/apikey).

Without a key, this part still runs on a recorded reply, so you can read the pipeline.

In [ ]:
import json
import pandas as pd

# Read the key from a Colab secret or an environment variable. Never from the code.
API_KEY = os.environ.get("GEMINI_API_KEY")
try:
    from google.colab import userdata          # only exists inside Colab
    API_KEY = API_KEY or userdata.get("GEMINI_API_KEY")
except Exception:
    pass
LIVE = bool(API_KEY)
print("live Gemini calls:", LIVE, "" if LIVE else "(no key found, using the recorded reply)")

**Annotation** is labelling. You give the model a fixed set of labels and a batch of items,
and ask for the labels back as JSON so a program can read them. Write the labels down first,
with a one-line definition each. That definition is the method.

The items: ten letters to *Dear Abby*, 1992 to 2017, from
[The Pudding's data](https://github.com/the-pudding/data/tree/master/dearabby). Three labels
per letter. Two you check against your own reading; the third, the decade, you check against
the truth.

In [ ]:
letters = [
 "is it ok to keep the ring after the engagement's been broken? my boyfriend wants the ring back, and i feel that it was a gift that is mine to keep. i need an answer fast, because he is fuming.",
 "how do you tell the difference between someone with a gambling problem and someone who is trying to become a poker champion? the person is my husband, and i'd like to support his dream of being a champion. i have never been around gamblers, and i am not sure where the line is drawn.",
 "i have a male friend who was raised with beautiful manners and always opens a door for a lady. the last time we spoke, he told me he had opened a door for a woman and she told him off! she said she didn't need any help. my friend didn't know what to say. i told him to just ignore what she said. was there a polite comeback for him?",
 "my wife and i work and lead busy lives. the dinner table is the only place we can sit together with our son and have a relaxing conversation. my wife, however, goes off and eats by herself saying she can't wait, even though dinner is almost ready. i have tried telling her i prefer family time, but she brushes me off or becomes angry. any suggestions?",
 "i was in and out of a relationship with bob for four years, and we recently split up again. last september i bought an airline ticket for him to accompany me on a florida vacation, but we broke up, so i cashed in his ticket. bob keeps calling me and saying he wants his present so he can go away. i said no way! was i wrong?",
 "i'm in middle school. i have had a few boyfriends since i started here. i try my best to look ok each day, but i always find a flaw in the way i look or act. sometimes i find it hard to trust guys when they tell me i'm pretty. can you please help me learn to trust people and be comfortable with my body?",
 "i am an adult male with a longtime problem. whether it's a sad or happy occasion, i start crying, sometimes sobbing. i try to avoid any situation that may cause this. i am at a new point in my life where i can no longer avoid these situations. please don't suggest i live with it. is there a magic pill to control this?",
 "after 31 years of marriage, my wife and i have split up. we love each other, but after the kids moved out we realized we have little in common. what is an appropriate christmas gift for an ex-wife? we are on friendly terms and will probably spend the holidays together with our children.",
 "i work and live in an ethnically and religiously diverse community. there is also a welcoming and open lgbt community here. while i was having lunch with a new employee, she mentioned that she was married. my first thought was that she was married to a woman, but later it sounded like her spouse was male. is it ever ok to ask the gender of someone's spouse?",
 "i have been playing the piano for five years and i still enjoy it. but over the past year and a half, going for lessons every week and having to practice is getting old for me. it's not the teacher, it's not my parents, it's me. my heart isn't in it anymore. what do you think i should do?",
]
years = [1992, 2007, 2009, 2010, 2010, 2011, 2012, 2013, 2016, 2017]

# Your reading, before the model sees anything.
by_hand = pd.DataFrame({
    "topic": ["romance", "money", "manners", "family", "romance", "self", "health", "manners", "manners", "self"],
    "wants": ["validation", "advice", "advice", "advice", "validation", "advice", "advice", "advice", "advice", "advice"],
})

PROMPT = """Label each letter written to an advice column.
"topic": one of romance, family, money, manners, self, health. The main thing the letter is about.
"wants": advice (the writer wants to be told what to do) or validation (the writer wants to be told they were right).
"decade": your best guess at when it was written, one of 1980s, 1990s, 2000s, 2010s. Use any clue in the wording.
Return ONLY a JSON list of objects with keys "topic", "wants", "decade", one per letter, in order.
Letters:
{items}"""

# Recorded reply so this runs with no key. A live call will label differently in places.
CASSETTE = json.dumps([
    {"topic": "romance", "wants": "validation", "decade": "1990s"},
    {"topic": "money",   "wants": "advice",     "decade": "2000s"},
    {"topic": "manners", "wants": "validation", "decade": "2000s"},
    {"topic": "family",  "wants": "advice",     "decade": "2010s"},
    {"topic": "money",   "wants": "validation", "decade": "2010s"},
    {"topic": "self",    "wants": "advice",     "decade": "2010s"},
    {"topic": "health",  "wants": "advice",     "decade": "2000s"},
    {"topic": "manners", "wants": "advice",     "decade": "2010s"},
    {"topic": "manners", "wants": "advice",     "decade": "2010s"},
    {"topic": "self",    "wants": "advice",     "decade": "2010s"},
])

def annotate(items):
    prompt = PROMPT.format(items="\n".join(f"{i+1}. {x}" for i, x in enumerate(items)))
    if LIVE:
        import google.generativeai as genai
        genai.configure(api_key=API_KEY)
        raw = genai.GenerativeModel("gemini-2.5-flash").generate_content(prompt).text
    else:
        raw = CASSETTE
    raw = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    return pd.DataFrame(json.loads(raw))

model_says = annotate(letters)

table = pd.DataFrame({
    "letter":      [l[:36] + "…" for l in letters],
    "topic you":   by_hand["topic"],   "topic model": model_says["topic"],
    "wants you":   by_hand["wants"],   "wants model": model_says["wants"],
    "written":     years,              "decade model": model_says["decade"],
})
print(table.to_string(index=False))

In [ ]:
true_decade = [f"{y // 10 * 10}s" for y in years]
print(f"topic   agrees with you on {(by_hand['topic'] == model_says['topic']).mean():.0%}")
print(f"wants   agrees with you on {(by_hand['wants'] == model_says['wants']).mean():.0%}")
print(f"decade  correct           {(model_says['decade'] == true_decade).mean():.0%}\n")

for i in range(len(letters)):
    flags = []
    if by_hand["topic"][i] != model_says["topic"][i]:
        flags.append(f"topic: you {by_hand['topic'][i]}, model {model_says['topic'][i]}")
    if by_hand["wants"][i] != model_says["wants"][i]:
        flags.append(f"wants: you {by_hand['wants'][i]}, model {model_says['wants'][i]}")
    if model_says["decade"][i] != true_decade[i]:
        flags.append(f"decade: really {years[i]}, guessed {model_says['decade'][i]}")
    if flags:
        print(f"{i+1:2d}. {letters[i][:60]}…")
        for f in flags:
            print("     ", f)

Read the disagreements before the percentages.

The door-opening letter: you said the writer wants advice, the model said validation. Both
readings are there in the text. She asks for a comeback, but she has already told her friend
what to do. The label set has no rule for that, so the model made one up. Fix the definition.

Bob's airline ticket: romance to you, money to the model. Also defensible. A single "topic"
label is a choice you made, and some letters refuse it.

The decade guesses are the model reading culture. Poker champion, a welcoming LGBT community,
a middle-schooler with boyfriends: each one dates a letter. Where it guesses wrong, ask which
cue it trusted.

### Images

The same model takes pictures. Hand it the 18 paintings from Part 6 and ask the same shape of
question: a fixed label set, a reason, and one guess you can mark.

CLIP already labelled these in Part 6 and got Bronzino's *Portrait of a Young Man* wrong. Now
ask a model that reads the picture instead of matching it to a phrase.

In [ ]:
# What the Met's own titles say. Read off the titles, not guessed.
sitter = ["man", "man", "woman", "man", "group", "man", "woman", "man", "man",
          "man", "man", "woman", "man", "man", "man", "group", "man", "man"]

IMAGE_PROMPT = """Look at this portrait painting and answer about the sitter.
"sitter": one of man, woman, group (two or more people).
"century": your best guess at when it was painted, like "16th" or "19th".
"tell": the single visual detail that decided your answer, under ten words.
Return ONLY one JSON object with keys "sitter", "century", "tell"."""

# Recorded replies so this runs with no key.
CASSETTE_IMG = json.dumps([
 {"sitter":"man","century":"19th","tell":"straw hat, beard, thick visible brushstrokes"},
 {"sitter":"man","century":"17th","tell":"aged face, dark cap, heavy chiaroscuro"},
 {"sitter":"woman","century":"16th","tell":"white linen headdress and wimple"},
 {"sitter":"man","century":"15th","tell":"white monastic hood, short beard"},
 {"sitter":"group","century":"15th","tell":"two figures framed at a stone window"},
 {"sitter":"man","century":"19th","tell":"dark tailcoat, high white collar"},
 {"sitter":"woman","century":"19th","tell":"dark dress, centre-parted hair"},
 {"sitter":"man","century":"17th","tell":"wide flat white collar, loose brushwork"},
 {"sitter":"man","century":"16th","tell":"black doublet, angular mannerist pose"},
 {"sitter":"man","century":"15th","tell":"red cap, dark cloak, three-quarter view"},
 {"sitter":"man","century":"16th","tell":"gloves held in one hand, flat green ground"},
 {"sitter":"woman","century":"17th","tell":"white ruff and dark Dutch dress"},
 {"sitter":"man","century":"16th","tell":"black cap and gown, letter in hand"},
 {"sitter":"man","century":"17th","tell":"ruff collar, dark doublet, oval format"},
 {"sitter":"man","century":"17th","tell":"long dark hair, plain collar, loose paint"},
 {"sitter":"group","century":"18th","tell":"painter at an easel with two onlookers"},
 {"sitter":"man","century":"15th","tell":"lined face, red hood, plain background"},
 {"sitter":"man","century":"15th","tell":"profile view, red cap, gold ground"},
])

def annotate_images(images):
    if LIVE:
        import google.generativeai as genai
        genai.configure(api_key=API_KEY)
        model = genai.GenerativeModel("gemini-2.5-flash")
        out = []
        for img in images:
            raw = model.generate_content([IMAGE_PROMPT, img]).text
            raw = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
            out.append(json.loads(raw))
        return pd.DataFrame(out)
    return pd.DataFrame(json.loads(CASSETTE_IMG))

seen = annotate_images(paintings)     # `paintings` comes from Part 6
seen.insert(0, "truth", sitter)
seen.insert(0, "painting", [t[:34] for t in titles])
seen.insert(3, "dated", made)
print(seen.to_string(index=False))

In [ ]:
right = (seen["truth"] == seen["sitter"]).mean()
print(f"sitter correct on {right:.0%}  (CLIP's zero-shot labels in Part 6 got the Bronzino wrong)\n")
for _, r in seen[seen["truth"] != seen["sitter"]].iterrows():
    print(f"  {r['painting']}: really {r['truth']}, model said {r['sitter']} ({r['tell']})")

bronzino = seen[seen["painting"].str.startswith("Portrait of a Young Man")].iloc[0]
print(f"\nBronzino: model says {bronzino['sitter']}, because {bronzino['tell']}")
print(f"          painted {bronzino['dated']}, model guessed {bronzino['century']} century")

Gemini gets the Bronzino. CLIP could only ask which of five phrases sat closest to the whole
picture; this model can look at the doublet and the pose and say why.

That is not a reason to trust it everywhere. It is slower, it costs money per image, and the
"tell" it gives you is a story written after the answer, not the reason for it. What the tell
is good for is spotting the cases where a right answer came from the wrong evidence.

### Audio

It takes sound too. Five recordings from Wikimedia Commons, all public domain, spanning
1894 to 1927: a music-hall song on wax cylinder, a brass band, a president, the first jazz
record ever released, and a late-twenties dance orchestra.

Same shape again. A fixed label set, a reason, and one guess you can mark: the year.

In [ ]:
# Public domain, from Wikimedia Commons. The year is when it was RECORDED.
RECORDINGS = [
    (1894, "Daisy Bell, sung by Edward M. Favor",
     "https://upload.wikimedia.org/wikipedia/commons/0/07/Daisy_Bell_sung_by_Edward_M._Favor_denoised.ogg"),
    (1904, "African Dreamland, Sousa's Band",
     "https://upload.wikimedia.org/wikipedia/commons/5/5d/African-Dreamland-Sousa_s-Band-_1904_-George-Atwater.ogg"),
    (1912, "Theodore Roosevelt, The Liberty of the People",
     "https://upload.wikimedia.org/wikipedia/commons/f/fd/Theodore_Roosevelt_%22The_liberty_of_the_people%22_speech.ogg"),
    (1917, "Livery Stable Blues, Original Dixieland Jass Band",
     "https://upload.wikimedia.org/wikipedia/commons/1/19/Original_Dixieland_Jass_Band_-_Livery_Stable_Blues_%281917%29_with_hiss_reduction.ogg"),
    (1927, "Rhythm Step, Fred Elizalde and his Orchestra",
     "https://upload.wikimedia.org/wikipedia/commons/8/8e/%22Rhythm_Step%22_-_Fred_Elizalde_and_his_Orchestra_%281927%29.opus"),
]

# Commons rate-limits hard. Skip whatever will not come down rather than stopping the notebook.
clips = []
for year, name, url in RECORDINGS:
    path = f"downloads/{year}{os.path.splitext(url)[1]}"
    if not os.path.exists(path):
        try:
            open(path, "wb").write(fetch(url).content)
        except Exception as e:
            os.path.exists(path) and os.remove(path)
            print(f"{year}  skipped, Commons said no ({type(e).__name__}). Re-run the cell later.")
            continue
    clips.append((year, name, path))
    print(f"{year}  {os.path.getsize(path)/1e6:.1f} MB  {name}")

print(f"\n{len(clips)} of {len(RECORDINGS)} recordings ready")
from IPython.display import Audio, display
if clips:
    display(Audio(clips[-1][2]))        # change the index and listen to another

In [ ]:
AUDIO_PROMPT = """Listen to this recording and answer.
"kind": one of song, band, speech.
"instruments": up to three you can hear, as a list of strings.
"year": your best guess at the year it was RECORDED, as a number.
"tell": the sound that dated it for you, under ten words.
Return ONLY one JSON object with keys "kind", "instruments", "year", "tell"."""

# Recorded replies so this runs with no key, keyed by year so any subset works.
CASSETTE_AUD = {
 1894: {"kind":"song","instruments":["male voice","piano","brass"],"year":1900,
        "tell":"heavy cylinder hiss, thin narrow-band sound"},
 1904: {"kind":"band","instruments":["cornet","trombone","bass drum"],"year":1905,
        "tell":"acoustic horn recording of a marching band"},
 1912: {"kind":"speech","instruments":["male voice"],"year":1915,
        "tell":"oratorical delivery, flat acoustic capture"},
 1917: {"kind":"band","instruments":["cornet","clarinet","trombone"],"year":1918,
        "tell":"collective improvisation and barnyard effects"},
 1927: {"kind":"band","instruments":["saxophone","piano","drums"],"year":1930,
        "tell":"smooth dance arrangement, electrical recording"},
}

def annotate_audio(clips):
    rows = []
    if LIVE:
        import google.generativeai as genai
        genai.configure(api_key=API_KEY)
        model = genai.GenerativeModel("gemini-2.5-flash")
    for year, name, path in clips:
        if LIVE:
            blob = {"mime_type": "audio/ogg",        # .ogg and .opus are both Ogg containers
                    "data": open(path, "rb").read()}
            raw = model.generate_content([AUDIO_PROMPT, blob]).text
            raw = raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
            rows.append(json.loads(raw))
        else:
            rows.append(CASSETTE_AUD[year])
    return pd.DataFrame(rows)

heard = annotate_audio(clips)
heard.insert(0, "recorded", [y for y, _, _ in clips])
heard.insert(1, "what it is", [n[:34] for _, n, _ in clips])
heard["instruments"] = heard["instruments"].apply(", ".join)
print(heard.to_string(index=False))

In [ ]:
off = (heard["year"] - heard["recorded"]).abs()
print(f"year guesses are off by {off.mean():.0f} years on average, worst {off.max()}\n")
for _, row in heard.iterrows():
    print(f"  {row['recorded']}  guessed {row['year']}  ({row['year'] - row['recorded']:+d})  {row['tell']}")

Instruments and kind it gets. The year it only approximates, and the errors lean late: an
acoustic recording from 1894 and one from 1917 sound equally "old" to a model, and old is a
wide band.

That is the useful result. Use a model for the label you can check, and a person for the one
you cannot. Never let a confident number into a table without marking it as a guess.

Three habits when a model labels for you:

1. Label a sample by hand first and report the agreement. Week 7 does this with 30 items.
2. Read every disagreement. Most of them are a definition that failed, not a model that did.
3. Count the calls. A corpus of 10,000 rows is 10,000 calls, or a few hundred if you batch
   them as with the letters above. Free tiers are rate-limited, and images and audio cost more
   per item than text.

## Next

- **Your own corpus.** Part 5 takes any list of strings.
- **A bigger embedder.** `all-mpnet-base-v2`, or the
  [MTEB leaderboard](https://huggingface.co/spaces/mteb/leaderboard).
- **Fine-tuning.** `cool-methods/finetune_modernbert.ipynb`, once an off-the-shelf model has
  failed you.
- **Annotation at scale.** `week07_annotator.ipynb`: the same call over a whole corpus, with a hand-labelled gold set to audit it against.
- **Other pipelines.** `"zero-shot-classification"`, `"ner"`, `"summarization"`,
  `"image-classification"`.

Three things to keep saying:

1. Name the model and version. "An AI said" is not a method.
2. A model trained on the open web knows the open web, not your corpus.
3. A confident score is not a correct answer.